In [1]:
import numpy as np
import pandas as pd

In [2]:
raport = pd.read_csv('raport_kwartalny_sprzedaz.csv')
katalog = pd.read_csv('katalog_produktow.csv')

In [5]:
raport.head(20)

,Data_Transakcji,ID_Produktu_Sprzedaz,Region,Ilosc_Sprzedana,Cena_Jednostkowa_Transakcji,Wartosc_Sprzedazy
0,2025-05-08,1074,Północ,1,44.30,44.30
1,2025-05-13,1068,Północ,5,126.45,632.25
2,2025-04-12,1090,Wschód,2,3974.94,7949.88
3,2025-05-27,1031,Zachód,1,230.14,230.14
4,2025-06-23,1026,Wschód,5,167.64,838.20
5,2025-04-30,1005,Północ,3,974.83,2924.49
6,2025-06-07,1048,Północ,5,4442.37,22211.85
7,2025-06-13,1062,Wschód,4,707.26,2829.04
8,2025-06-06,1044,Południe,2,2079.38,4158.76
9,2025-04-16,1053,Północ,2,222.64,445.28


In [4]:
katalog.head(20)

,ID_Produktu_Katalog,Nazwa_Produktu,Cena_Katalogowa,Kategoria_Produktu
0,1001,Mieć na imię Ósmy,339.36,Akcesoria
1,1002,Mało Zapalenie,153.68,Biuro
2,1003,Kalendarz Głos,291.73,Oprogramowanie
3,1004,Dzielnica Iść,307.01,Akcesoria
4,1005,Lot Centralny,977.54,Elektronika
5,1006,Skrzydło Jedenaście,54.96,Biuro
6,1007,Kostka Świecić,24.55,Biuro
7,1008,Organ Model,116.40,Akcesoria
8,1009,Świat Statek,305.78,Oprogramowanie
9,1010,Klasa Oficjalny,2686.34,AGD


In [23]:
nieistniejace_id = raport[~raport['ID_Produktu_Sprzedaz'].isin(katalog['ID_Produktu_Katalog'])]
liczba_unikalnych_niepasujacych = nieistniejace_id['ID_Produktu_Sprzedaz'].nunique()

print("Liczba unikalnych transakcji bez odpowiednika w katalogu:", liczba_unikalnych_niepasujacych)

Liczba unikalnych transakcji bez odpowiednika w katalogu: 146


In [24]:
merged_dfs =  pd.merge(raport, katalog, how='inner', left_on='ID_Produktu_Sprzedaz', right_on='ID_Produktu_Katalog')
wartosc_sprzedazy = sum(merged_dfs['Wartosc_Sprzedazy'])
print('Cała wartość udanych transakcji:', round(wartosc_sprzedazy,2))

Cała wartość udanych transakcji: 31336539.21


In [29]:
polaczone = pd.merge(raport, katalog, left_on='ID_Produktu_Sprzedaz',right_on='ID_Produktu_Katalog',how='inner')

grupa = polaczone.groupby('ID_Produktu_Sprzedaz').agg({
    'Cena_Jednostkowa_Transakcji': 'mean',
    'Cena_Katalogowa': 'first',
    'Nazwa_Produktu': 'first'
}).reset_index()

grupa['roznica_procentowa'] = ((grupa['Cena_Jednostkowa_Transakcji'] - grupa['Cena_Katalogowa']).abs()/ grupa['Cena_Katalogowa']) * 100
najwieksza_roznica = grupa.sort_values(by='roznica_procentowa', ascending=False).iloc[0]
najwieksza_roznica = grupa.loc[grupa['roznica_procentowa'].abs().idxmax()]
znak = "+" if najwieksza_roznica['roznica_procentowa'] > 0 else "-"
roznica_zaokraglona = round(abs(najwieksza_roznica['roznica_procentowa']), 2)

print("nazwa:", najwieksza_roznica['Nazwa_Produktu'])
print(f"różnica: {znak}{roznica_zaokraglona}%")


Nazwa_Produktu: Miara Przyczyna
Różnica procentowa: +30.0%


In [37]:
# Ile wierszy w raporcie sprzedaży ma 
# niezgodność między Wartosc_Sprzedazy a iloczynem Ilosc_Sprzedana * Cena_Jednostkowa_Transakcji?

wartosc_sprzedazy = raport['Wartosc_Sprzedazy']

raport['iloczyn_sprzedazy'] = raport['Ilosc_Sprzedana'] * raport['Cena_Jednostkowa_Transakcji']

raport[wartosc_sprzedazy != raport['iloczyn_sprzedazy']]

,Data_Transakcji,ID_Produktu_Sprzedaz,Region,Ilosc_Sprzedana,Cena_Jednostkowa_Transakcji,Wartosc_Sprzedazy,iloczyn_sprzedazy
4,2025-06-23,1026,Wschód,5,167.64,838.20,838.20
5,2025-04-30,1005,Północ,3,974.83,2924.49,2924.49
11,2025-04-27,1006,Wschód,3,52.64,157.92,157.92
16,2025-05-14,1038,Zachód,1,4919.60,4915.53,4919.60
19,2025-04-03,1012,Południe,5,103.57,517.85,517.85
...,...,...,...,...,...,...,...
7486,2025-04-27,1023,Zachód,1,2924.73,2922.17,2924.73
7492,2025-04-04,1008,Zachód,3,112.15,336.45,336.45
7493,2025-06-26,1001,Wschód,3,333.70,1001.10,1001.10
7496,2025-06-24,1067,Zachód,5,752.41,3762.05,3762.05


In [40]:
wartosc_obliczona = raport['Ilosc_Sprzedana'] * raport['Cena_Jednostkowa_Transakcji']
niezgodnosc_mask = ~np.isclose(raport['Wartosc_Sprzedazy'], wartosc_obliczona, atol=0.01)
liczba_niezgodnych_wierszy = niezgodnosc_mask.sum()

print("liczba wierszy:", liczba_niezgodnych_wierszy)

Liczba wierszy z niezgodnością wartości sprzedaży: 368
